# Try to use the substations as origins and highway on / of ramps as destinations

Do your imports

In [1]:
# === Standard Library ===
from pathlib import Path
import pickle

# === Scientific & Data Libraries ===
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm


# === Geospatial Libraries ===
import geopandas as gpd
import rasterio
import folium
from shapely.geometry import box, Point, LineString, Polygon, shape
from pyproj import Transformer
import networkx as nx
import geohexgrid as ghg
from shapely.ops import transform


# === RA2CE Project Imports ===
from ra2ce.network.network_config_data.enums.aggregate_wl_enum import AggregateWlEnum
from ra2ce.network.network_config_data.enums.source_enum import SourceEnum
from ra2ce.network.network_config_data.enums.network_type_enum import NetworkTypeEnum
from ra2ce.network.network_config_data.enums.road_type_enum import RoadTypeEnum
from ra2ce.network.network_config_data.network_config_data import (
    HazardSection,
    NetworkConfigData,
    NetworkSection,
    OriginsDestinationsSection
)
from ra2ce.network.exporters.geodataframe_network_exporter import GeoDataFrameNetworkExporter
from ra2ce.network.exporters.multi_graph_network_exporter import MultiGraphNetworkExporter
from ra2ce.network.network_wrappers.osm_network_wrapper.osm_network_wrapper import OsmNetworkWrapper
from ra2ce.ra2ce_handler import Ra2ceHandler

import os
from pathlib import Path


c:\python\ra2ce\ra2ce_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# specify the name of the path to the project folder where you created the RA2CE folder setup

root_dir = Path(r'C:\python\powerpath\data')
#root_dir = Path.cwd().parent.joinpath("data")
assert root_dir.exists()
static_path = root_dir.joinpath("static")
hazard_path = static_path.joinpath("hazard")
network_path = static_path.joinpath("network")
output_path = static_path.joinpath("output_graph")


In [3]:
# some preliminary functions

def get_all_files(directory: str) -> list[Path]:
    p = Path(directory)
    return [file for file in p.iterdir() if file.is_file()]

def read_pickle(file_path: str):
    with open(file_path, 'rb') as file:
        data = pickle.load(file)
    return data

def read_gpkg_to_gdf(file_path: str, layer: str = None) -> gpd.GeoDataFrame:
    # Read the geopackage file into a GeoDataFrame
    gdf = gpd.read_file(file_path, layer=layer)
    return gdf

In [4]:
hazard_files = get_all_files(hazard_path)
hazard_crs = "EPSG:4326" # for the hackathon case => "EPSG:4326" 

for hazard_file in hazard_files:
    print (hazard_file)

C:\python\powerpath\data\static\hazard\delfland_ghg200m_wgs84.tif


## Find the study area

<font color='blue'>To do in a later stage: make this flexible based on hazard map</font> 

In [5]:
Extent_path = network_path.joinpath("try_study_area_larger.shp")
Extent = gpd.read_file(Extent_path, driver='ESRI Shapefile')
shapely_polygon = Extent.geometry.iloc[0]

# Preprocessing origins and destinations - to be transferred to seperate notebook

In [6]:
highways = gpd.read_file(r"C:\python\BRS_RWS\Wegenbestand\Rijkswegen_uit_nwb\rijkswegen.shp", driver='ESRI Shapefile')

In [7]:
# Filter for Rijkswegen
highways_rijkswegen = highways[highways['WEGBEHSRT'] == 'R']

# Filter for on- and off-ramps
op_en_afritten = highways_rijkswegen[
    (highways_rijkswegen['HECTO_LTTR'] == 'd') | (highways_rijkswegen['HECTO_LTTR'] == 'a')
].copy()

# Add category column based on HECTO_LTTR
op_en_afritten["category"] = op_en_afritten["HECTO_LTTR"].map({
    "d": "oprit",
    "a": "afrit"
})

# Create a new GeoDataFrame with centroid as geometry
op_en_afritten_points = op_en_afritten.copy()
op_en_afritten_points.set_geometry(op_en_afritten_points.geometry.centroid, inplace=True)

# Keep only necessary columns
op_en_afritten_points = op_en_afritten_points[["id", "geometry", "category"]].copy()

# Reset index and assign it to 'id'
op_en_afritten_points.reset_index(drop=True, inplace=True)
op_en_afritten_points["id"] = op_en_afritten_points.index


In [8]:
op_en_afritten_points.explore(column='category')

In [9]:
substations_read = gpd.read_file(r"C:\python\powerpath\data\electricity\msls_stations_clipped.shp", driver='ESRI Shapefile')

In [10]:
substations_read["centroid"] = substations_read.geometry.centroid
substations_points = substations_read.set_geometry("centroid")
substations = substations_points[["centroid"]].copy()
substations = substations.rename(columns={"centroid": "geometry"})
substations.set_geometry("geometry", inplace=True)

C:\Users\meije_le\AppData\Local\Temp\ipykernel_46036\3671779784.py:1: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  substations_read["centroid"] = substations_read.geometry.centroid


In [11]:
substations.explore()

# Refactored modular script

In [12]:
# accessibility_utils.py

import pickle
import networkx as nx
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import Point
from pyproj import Transformer
from tqdm import tqdm

def load_graph(path):
    with open(path, "rb") as f:
        return pickle.load(f)

def project_graph_coords(G, from_crs, to_crs):
    transformer = Transformer.from_crs(from_crs, to_crs, always_xy=True)
    for n, d in G.nodes(data=True):
        d["x_m"], d["y_m"] = transformer.transform(d["x"], d["y"])
    for u, v, d in G.edges(data=True):
        x1, y1 = G.nodes[u]["x_m"], G.nodes[u]["y_m"]
        x2, y2 = G.nodes[v]["x_m"], G.nodes[v]["y_m"]
        d["length"] = ((x2 - x1)**2 + (y2 - y1)**2)**0.5
    return G

def filter_motorway_edges(G):
    G_filtered = G.copy()
    edges_to_remove = [
        (u, v) for u, v, d in G_filtered.edges(data=True)
        if "highway" in d and isinstance(d["highway"], str) and "motorway" in d["highway"].lower()
    ]
    G_filtered.remove_edges_from(edges_to_remove)
    G_filtered.remove_nodes_from(list(nx.isolates(G_filtered)))
    return G_filtered

def filter_hazard_graph(G, threshold, hazard_column):
    def is_motorway(highway):
        if isinstance(highway, str):
            return "motorway" in highway.lower()
        elif isinstance(highway, list):
            return any("motorway" in str(h).lower() for h in highway)
        return False

    def is_protected(d):
        return pd.notna(d.get("bridge")) or pd.notna(d.get("tunnel"))

    G_filtered = G.copy()
    edges_to_remove = [
        (u, v) for u, v, d in G_filtered.edges(data=True)
        if d.get(hazard_column, 0) > threshold and not is_motorway(d.get("highway")) and not is_protected(d)
    ]
    G_filtered.remove_edges_from(edges_to_remove)
    G_filtered.remove_nodes_from(list(nx.isolates(G_filtered)))
    return G_filtered

def build_node_gdf(G, crs):
    return gpd.GeoDataFrame(
        [{"node": n, "geometry": Point(d["x_m"], d["y_m"])} for n, d in G.nodes(data=True)],
        crs=crs
    )

def get_nearest_node(point, node_gdf, transformer, max_distance):
    if transformer is not None:
        x, y = transformer.transform(point.x, point.y)
        point_rd = gpd.GeoSeries([Point(x, y)], crs=node_gdf.crs)
    else:
        point_rd = gpd.GeoSeries([point], crs=node_gdf.crs)
    try:
        nearest = gpd.sjoin_nearest(gpd.GeoDataFrame(geometry=point_rd), node_gdf, how="left", max_distance=max_distance)
        return nearest.iloc[0]["node"]
    except:
        return None

def assign_nearest_nodes(points_gdf, node_gdf, transformer, max_distance):
    points_gdf["nearest_node"] = points_gdf.geometry.apply(
        lambda pt: get_nearest_node(pt, node_gdf, transformer, max_distance)
    )
    return points_gdf

def compute_highway_accessibility(origins, highway_nodes, G, label_prefix, max_targets=5, cutoff=10000):
    enriched_rows = []

    for _, row in tqdm(origins.iterrows(), total=len(origins)):
        origin_id = row["id"]
        origin_node = row["nearest_node"]
        geometry = row.geometry

        is_isolated = origin_node is None or origin_node not in G

        if is_isolated:
            enriched_rows.append({
                "id": origin_id,
                "geometry": geometry,
                f"can_reach_highway_{label_prefix}": False,
                f"can_be_reached_from_highway_{label_prefix}": False,
                f"min_dist_to_highway_{label_prefix}": np.nan,
                f"min_dist_from_highway_{label_prefix}": np.nan,
                f"is_isolated_{label_prefix}": True
            })
            continue

        try:
            lengths_to = nx.single_source_dijkstra_path_length(G, source=origin_node, weight="length", cutoff=cutoff)
            reachable_highways_to = {n: d for n, d in lengths_to.items() if n in highway_nodes}
        except:
            reachable_highways_to = {}

        sorted_to = sorted(reachable_highways_to.values())[:max_targets] if reachable_highways_to else []

        distances_from = []
        for hw_node in list(highway_nodes)[:max_targets]:
            try:
                lengths_from = nx.single_source_dijkstra_path_length(G, source=hw_node, weight="length", cutoff=cutoff)
                if origin_node in lengths_from:
                    distances_from.append(lengths_from[origin_node])
            except:
                continue

        enriched_rows.append({
            "id": origin_id,
            "geometry": geometry,
            f"can_reach_highway_{label_prefix}": len(sorted_to) > 0,
            f"can_be_reached_from_highway_{label_prefix}": len(distances_from) > 0,
            f"min_dist_to_highway_{label_prefix}": min(sorted_to) if sorted_to else np.nan,
            f"min_dist_from_highway_{label_prefix}": min(distances_from) if distances_from else np.nan,
            f"is_isolated_{label_prefix}": False
        })

    return gpd.GeoDataFrame(enriched_rows, geometry="geometry", crs=origins.crs)

def compute_directional_accessibility(origins, destinations, G, label_prefix, max_targets=5, cutoff=10000):
    destination_nodes = set(destinations["nearest_node"].dropna())

    enriched_rows = []

    for _, row in tqdm(origins.iterrows(), total=len(origins)):
        origin_id = row["id"]
        origin_node = row["nearest_node"]
        geometry = row.geometry

        is_isolated = origin_node is None or origin_node not in G

        if is_isolated:
            enriched_rows.append({
                "id": origin_id,
                "geometry": geometry,
                f"can_reach_{label_prefix}": False,
                f"min_dist_to_{label_prefix}": np.nan,
                f"is_isolated_{label_prefix}": True
            })
            continue

        try:
            lengths = nx.single_source_dijkstra_path_length(G, source=origin_node, weight="length", cutoff=cutoff)
            reachable = {n: d for n, d in lengths.items() if n in destination_nodes}
        except:
            reachable = {}

        sorted_dists = sorted(reachable.values())[:max_targets] if reachable else []

        enriched_rows.append({
            "id": origin_id,
            "geometry": geometry,
            f"can_reach_{label_prefix}": len(sorted_dists) > 0,
            f"min_dist_to_{label_prefix}": min(sorted_dists) if sorted_dists else np.nan,
            f"is_isolated_{label_prefix}": False
        })

    return gpd.GeoDataFrame(enriched_rows, geometry="geometry", crs=origins.crs)


In [16]:
# === CRS ===
from_crs = "EPSG:4326"
to_crs = "EPSG:28992"

# === PATHS ===
road_network_path = output_path / "base_graph.p"
hazard_graph_path = output_path / "base_graph_hazard.p"
flood_tiff_path = hazard_file
comparison_output_path = output_path / "highway_access_comparison.gpkg"

# === Load and project graph ===
print("Loading and projecting road network...")
G = project_graph_coords(load_graph(road_network_path), from_crs, to_crs)
G_filtered = filter_motorway_edges(G.copy())  # for snapping only
G_hazard = filter_hazard_graph(G.copy(), threshold=0.3, hazard_column="EV1_ma")

node_gdf = build_node_gdf(G_filtered, crs=to_crs)     # for snapping substations
node_gdf_full = build_node_gdf(G, crs=to_crs)          # for snapping ramps

# === Prepare data ===
print("Preparing input datasets...")
substations = substations.reset_index(drop=True)
substations["id"] = substations.index

ramps = op_en_afritten_points.copy()
ramps["id"] = ramps.index
oprits = ramps[ramps["category"] == "oprit"].copy()
oprits["id"] = oprits.index
afrits = ramps[ramps["category"] == "afrit"].copy()
afrits["id"] = afrits.index

# === Assign nearest nodes ===
print("Assigning nearest nodes to substations (non-motorway)...")
substations = assign_nearest_nodes(substations, node_gdf, transformer=None, max_distance=500)

print("Assigning nearest nodes to ramps (motorway-inclusive)...")
oprits = assign_nearest_nodes(oprits, node_gdf_full, transformer=None, max_distance=500)
afrits = assign_nearest_nodes(afrits, node_gdf_full, transformer=None, max_distance=500)
ramps = assign_nearest_nodes(ramps, node_gdf_full, transformer=None, max_distance=500)

# === Add isolation flags ===
print("Flagging isolated nodes...")
substations["is_isolated"] = substations["nearest_node"].isna() | ~substations["nearest_node"].isin(G.nodes)
oprits["is_isolated"] = oprits["nearest_node"].isna() | ~oprits["nearest_node"].isin(G.nodes)
afrits["is_isolated"] = afrits["nearest_node"].isna() | ~afrits["nearest_node"].isin(G.nodes)

# === Directional accessibility ===
print("Computing directional accessibility: substations → oprit...")
access_to_highway = compute_directional_accessibility(
    origins=substations,
    destinations=oprits,
    G=G,
    label_prefix="oprit"
)

print("Computing directional accessibility: afrit → substations...")
access_from_highway = compute_directional_accessibility(
    origins=afrits,
    destinations=substations,
    G=G,
    label_prefix="substations"
)

# === Merge results ===
print("Merging directional accessibility results...")
substations_access = access_to_highway
afrits_access = access_from_highway

# === General accessibility (hazard comparison) ===
print("Reassigning nearest nodes to substations for hazard comparison...")
origins = substations_access.copy()
origins = assign_nearest_nodes(origins, node_gdf, transformer=None, max_distance=500)
origins["is_isolated_hazard"] = origins["nearest_node"].isna() | ~origins["nearest_node"].isin(G_hazard.nodes)


# === Assign ramp nodes again for hazard comparison ===
print("Preparing highway node sets...")
oprit_nodes = set(oprits["nearest_node"].dropna())
afrit_nodes = set(afrits["nearest_node"].dropna())
highway_nodes = oprit_nodes.union(afrit_nodes)

# === Accessibility comparison ===
print("Computing accessibility without hazard...")
access_no_hazard = compute_highway_accessibility(origins, highway_nodes, G, label_prefix="nohazard")

print("Computing accessibility with hazard...")
access_hazard = compute_highway_accessibility(origins, highway_nodes, G_hazard, label_prefix="hazard")

# === Merge and compare ===
print("Merging and comparing accessibility results...")
access_df = access_no_hazard.merge(access_hazard.drop(columns="geometry"), on="id")
access_df["dist_to_diff"] = access_df["min_dist_to_highway_hazard"] - access_df["min_dist_to_highway_nohazard"]
access_df["dist_from_diff"] = access_df["min_dist_from_highway_hazard"] - access_df["min_dist_from_highway_nohazard"]

# === Export ===
print(f"Saving results to: {comparison_output_path}")
access_df.to_file(comparison_output_path)
print("✅ Accessibility comparison completed and saved.")


Loading and projecting road network...
Preparing input datasets...
Assigning nearest nodes to substations (non-motorway)...
Assigning nearest nodes to ramps (motorway-inclusive)...
Flagging isolated nodes...
Computing directional accessibility: substations → oprit...


100%|██████████| 1258/1258 [00:00<00:00, 19342.19it/s]


Computing directional accessibility: afrit → substations...


100%|██████████| 1835/1835 [00:00<00:00, 3528.51it/s]


Merging directional accessibility results...
Reassigning nearest nodes to substations for hazard comparison...
Preparing highway node sets...
Computing accessibility without hazard...


100%|██████████| 1258/1258 [00:00<00:00, 19702.89it/s]


Computing accessibility with hazard...


100%|██████████| 1258/1258 [00:00<00:00, 19940.04it/s]

Merging and comparing accessibility results...
Saving results to: C:\python\powerpath\data\static\output_graph\highway_access_comparison.gpkg


✅ Accessibility comparison completed and saved.


In [17]:
access_df

,id,geometry,can_reach_highway_nohazard,can_be_reached_from_highway_nohazard,min_dist_to_highway_nohazard,min_dist_from_highway_nohazard,is_isolated_nohazard,can_reach_highway_hazard,can_be_reached_from_highway_hazard,min_dist_to_highway_hazard,min_dist_from_highway_hazard,is_isolated_hazard,dist_to_diff,dist_from_diff
0,0,POINT (4.28845 52.03010),False,False,NaN,NaN,True,False,False,NaN,NaN,True,NaN,NaN
1,1,POINT (4.29269 52.03074),False,False,NaN,NaN,True,False,False,NaN,NaN,True,NaN,NaN
2,2,POINT (4.29834 52.03123),False,False,NaN,NaN,True,False,False,NaN,NaN,True,NaN,NaN
3,3,POINT (4.30326 52.03124),False,False,NaN,NaN,True,False,False,NaN,NaN,True,NaN,NaN
4,4,POINT (4.29652 52.03228),False,False,NaN,NaN,True,False,False,NaN,NaN,True,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1253,1253,POINT (4.34929 52.09185),False,False,NaN,NaN,True,False,False,NaN,NaN,True,NaN,NaN
1254,1254,POINT (4.35162 52.09290),False,False,NaN,NaN,True,False,False,NaN,NaN,True,NaN,NaN
1255,1255,POINT (4.30919 52.09238),False,False,NaN,NaN,True,False,False,NaN,NaN,True,NaN,NaN
1256,1256,POINT (4.31857 52.09278),False,False,NaN,NaN,True,False,False,NaN,NaN,True,NaN,NaN
